In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler
import pandas as pd
from scipy.spatial import cKDTree

In [2]:
# 1. Load model
model = keras.models.load_model("./models/final_model_outputS%.keras")
scaler = StandardScaler()

In [3]:
inputs = np.load("./inputs_stacked/inputs.npy")

In [4]:
X_full_scaled = scaler.fit_transform(inputs.reshape(-1, inputs.shape[-1])).reshape(inputs.shape)

In [13]:
from tqdm import tqdm

def batch_generator(X, batch_size=1024):
    N = X.shape[0]
    for i in range(0, N, batch_size):
        yield X[i:i+batch_size]

# Example: memory-efficient prediction
preds_list = []
batch_size = 1024  # adjust smaller if needed

for batch in tqdm(batch_generator(X_full_scaled, batch_size=batch_size), total=int(np.ceil(X_full_scaled.shape[0]/batch_size))):
    preds = model.predict(batch, batch_size=batch.shape[0], verbose=0)
    preds_list.append(preds)

# Concatenate all batches into final predictions
predictions = np.concatenate(preds_list, axis=0)
print(predictions.shape)

100%|██████████| 2758/2758 [02:33<00:00, 17.98it/s]

(2823300, 1)


In [14]:
print("Min prediction:", predictions.min())
print("Max prediction:", predictions.max())

Min prediction: 8.976355
Max prediction: 89.70504


In [15]:
predictions = predictions.flatten()

In [19]:
df = pd.read_csv("points.csv")

In [20]:
df['S%_values'] = predictions

In [21]:
df.head()

,X,Y,S%_values
0,3.343779,18.967783,85.915894
1,3.352772,18.967783,85.915894
2,3.361765,18.967783,86.852325
3,3.370759,18.967783,86.070168
4,3.379752,18.967783,86.070206


In [13]:
# df has columns: X, Y, pH_value
points = df[['X', 'Y']].values
values = df['S%_value'].values

# Find indices where pH < 4 and pH >= 4
valid_idx = np.where(values <= 100)[0]
high_idx = np.where(values > 100)[0]

# Build KDTree for points with pH >= 4
tree = cKDTree(points[valid_idx])

# For each low pH point, find nearest high pH point
distances, nearest_idx = tree.query(points[high_idx], k=1)

# Replace low values with the corresponding nearest high pH value
values[high_idx] = values[valid_idx[nearest_idx]]
df = df.drop(columns=["S%_value"])
# Update DataFrame
df['S%_values'] = values

In [14]:
# Quick check of pH range
min_S = df['S%_values'].min()
max_S = df['S%_values'].max()

print(f"Minimum S value: {min_S}")
print(f"Maximum S value: {max_S}")


Minimum S value: 0.0002269744873046875
Maximum S value: 99.99971008300781


In [22]:
df.to_csv('./predictions/S%.csv', index=False)